In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

In [2]:
DATA_DIR = Path("../data/raw")

defacto_path = DATA_DIR / "defacto_all.csv"
panel_path = DATA_DIR / "country_year_final_panel_small_new.csv"
output_path = DATA_DIR / "country_year_final_panel_small_new_defacto.csv"
audit_country_year_path = DATA_DIR / "defacto_unmatched_country_years.csv"
audit_summary_path = DATA_DIR / "defacto_unmatched_country_summary.csv"

In [3]:
df_defacto = pd.read_csv(defacto_path)
df_panel = pd.read_csv(panel_path)

print("Defacto shape:", df_defacto.shape)
print("Panel shape:", df_panel.shape)

display(df_defacto.head())
display(df_panel.head())

Defacto shape: (28092, 5)
Panel shape: (8580, 12)


,country_name,country_text_id,country_id,year,v2x_freexp
0,Mexico,MEX,3,1789,0.188
1,Mexico,MEX,3,1790,0.188
2,Mexico,MEX,3,1791,0.188
3,Mexico,MEX,3,1792,0.188
4,Mexico,MEX,3,1793,0.188


,Unnamed: 0,COUNTRY,iso3,year,dj_expression,wdj_expression,wdj_citizen,wdj_intermediaries,wdj_press,wdj_govprot,wdj_restriction,wdj_obligation
0,1,ASEAN,ASEAN,1894,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,ASEAN,ASEAN,1895,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,ASEAN,ASEAN,1896,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,ASEAN,ASEAN,1897,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,ASEAN,ASEAN,1898,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
defacto_vars = [
    "country_name",
    "country_text_id",
    "country_id",
    "year",
    "v2x_freexp",
]

panel_vars = [
    "COUNTRY",
    "iso3",
    "year",
    "dj_expression",
    "wdj_expression",
]

df_defacto_small = df_defacto[defacto_vars].copy()
df_panel_small = df_panel[panel_vars].copy()

df_defacto_small["country_text_id"] = df_defacto_small["country_text_id"].astype(str).str.strip().str.upper()
df_panel_small["iso3"] = df_panel_small["iso3"].astype(str).str.strip().str.upper()

df_defacto_small["year"] = df_defacto_small["year"].astype(int)
df_panel_small["year"] = df_panel_small["year"].astype(int)

In [5]:
defacto_key = ["country_text_id", "year"]
panel_key = ["iso3", "year"]

defacto_duplicates = df_defacto_small.duplicated(defacto_key).sum()
panel_duplicates = df_panel_small.duplicated(panel_key).sum()

print("Defacto duplicate country-years:", defacto_duplicates)
print("Panel duplicate country-years:", panel_duplicates)

assert defacto_duplicates == 0, "defacto_all has duplicate country-year keys"
assert panel_duplicates == 0, "country_year_final_panel_small_new has duplicate country-year keys"

Defacto duplicate country-years: 0
Panel duplicate country-years: 0


In [6]:
defacto_coverage = (
    df_defacto_small.groupby("country_text_id", as_index=False)
    .agg(
        defacto_country_name=("country_name", "first"),
        defacto_first_year=("year", "min"),
        defacto_last_year=("year", "max"),
        defacto_years=("year", "count"),
    )
    .rename(columns={"country_text_id": "iso3"})
)

aggregate_or_noncountry_codes = ["ASEAN", "AU", "EU", "OAS"]

In [7]:
df_merged = df_panel_small.merge(
    df_defacto_small,
    left_on=panel_key,
    right_on=defacto_key,
    how="left",
    validate="one_to_one",
)

df_merged = df_merged[[
    "COUNTRY",
    "iso3",
    "year",
    "dj_expression",
    "wdj_expression",
    "v2x_freexp",
    "country_name",
    "country_id",
]]

df_merged = df_merged.merge(defacto_coverage, on="iso3", how="left")
df_merged["defacto_merge_status"] = "matched"
df_merged["defacto_unmatched_reason"] = pd.NA

missing_defacto = df_merged["v2x_freexp"].isna()
no_defacto_entity = missing_defacto & df_merged["iso3"].isin(aggregate_or_noncountry_codes)
before_defacto_coverage = (
    missing_defacto
    & df_merged["defacto_first_year"].notna()
    & (df_merged["year"] < df_merged["defacto_first_year"])
)
after_defacto_coverage = (
    missing_defacto
    & df_merged["defacto_last_year"].notna()
    & (df_merged["year"] > df_merged["defacto_last_year"])
)
specific_year_gap = (
    missing_defacto
    & df_merged["defacto_first_year"].notna()
    & ~before_defacto_coverage
    & ~after_defacto_coverage
)
missing_unknown = missing_defacto & df_merged["defacto_first_year"].isna() & ~no_defacto_entity

df_merged.loc[no_defacto_entity, "defacto_merge_status"] = "unmatched_expected_no_defacto_entity"
df_merged.loc[no_defacto_entity, "defacto_unmatched_reason"] = "aggregate/non-country code not in defacto data"
df_merged.loc[before_defacto_coverage, "defacto_merge_status"] = "unmatched_expected_before_defacto_coverage"
df_merged.loc[before_defacto_coverage, "defacto_unmatched_reason"] = "panel year before defacto coverage starts"
df_merged.loc[after_defacto_coverage, "defacto_merge_status"] = "unmatched_expected_after_defacto_coverage"
df_merged.loc[after_defacto_coverage, "defacto_unmatched_reason"] = "panel year after defacto coverage ends"
df_merged.loc[specific_year_gap, "defacto_merge_status"] = "unmatched_specific_year_gap"
df_merged.loc[specific_year_gap, "defacto_unmatched_reason"] = "matched country key, but defacto value missing for this year"
df_merged.loc[missing_unknown, "defacto_merge_status"] = "unmatched_missing_defacto_country_key"
df_merged.loc[missing_unknown, "defacto_unmatched_reason"] = "country key not present in defacto data"

df_merged = df_merged[[
    "COUNTRY",
    "iso3",
    "year",
    "dj_expression",
    "wdj_expression",
    "v2x_freexp",
    "defacto_merge_status",
    "defacto_unmatched_reason",
    "country_name",
    "country_id",
    "defacto_first_year",
    "defacto_last_year",
]]

match_rate = df_merged["v2x_freexp"].notna().mean()
print(f"Merged shape: {df_merged.shape[0]:,} rows x {df_merged.shape[1]:,} columns")
print(f"v2x_freexp match rate: {match_rate:.2%}")

df_merged.head()

Merged shape: 8,580 rows x 12 columns
v2x_freexp match rate: 81.17%


,COUNTRY,iso3,year,dj_expression,wdj_expression,v2x_freexp,defacto_merge_status,defacto_unmatched_reason,country_name,country_id,defacto_first_year,defacto_last_year
0,ASEAN,ASEAN,1894,0.0,0.0,NaN,unmatched_expected_no_defacto_entity,aggregate/non-country code not in defacto data,NaN,NaN,NaN,NaN
1,ASEAN,ASEAN,1895,0.0,0.0,NaN,unmatched_expected_no_defacto_entity,aggregate/non-country code not in defacto data,NaN,NaN,NaN,NaN
2,ASEAN,ASEAN,1896,0.0,0.0,NaN,unmatched_expected_no_defacto_entity,aggregate/non-country code not in defacto data,NaN,NaN,NaN,NaN
3,ASEAN,ASEAN,1897,0.0,0.0,NaN,unmatched_expected_no_defacto_entity,aggregate/non-country code not in defacto data,NaN,NaN,NaN,NaN
4,ASEAN,ASEAN,1898,0.0,0.0,NaN,unmatched_expected_no_defacto_entity,aggregate/non-country code not in defacto data,NaN,NaN,NaN,NaN


In [8]:
unmatched = (
    df_merged.loc[df_merged["v2x_freexp"].isna(), ["COUNTRY", "iso3", "year"]]
    .groupby(["COUNTRY", "iso3"], as_index=False)
    .agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        unmatched_years=("year", "count"),
    )
    .sort_values(["unmatched_years", "COUNTRY"], ascending=[False, True])
)

print("Unmatched country-year rows:", int(df_merged["v2x_freexp"].isna().sum()))
unmatched.head(30)

Unmatched country-year rows: 1616


,COUNTRY,iso3,first_year,last_year,unmatched_years
0,ASEAN,ASEAN,1894,2025,132
1,African Union,AU,1894,2025,132
13,European Union,EU,1894,2025,132
29,Organization of American States,OAS,1894,2025,132
37,South Sudan,SSD,1894,2010,117
20,Kosovo,XKX,1894,1998,105
4,Azerbaijan,AZE,1894,1989,96
18,Kazakhstan,KAZ,1894,1989,96
40,Tajikistan,TJK,1894,1989,96
5,Bangladesh,BGD,1894,1970,77


In [9]:
df_merged.to_csv(output_path, index=False)
print(f"Saved merged country-year panel to {output_path}")

Saved merged country-year panel to ../data/raw/country_year_final_panel_small_new_defacto.csv


In [10]:
unmatched_rows = df_merged[df_merged["v2x_freexp"].isna()].copy()

panel_coverage = (
    df_panel_small.groupby(["COUNTRY", "iso3"], as_index=False)
    .agg(
        panel_first_year=("year", "min"),
        panel_last_year=("year", "max"),
        panel_years=("year", "count"),
    )
)

unmatched_audit = (
    unmatched_rows.groupby(["COUNTRY", "iso3"], as_index=False)
    .agg(
        first_unmatched_year=("year", "min"),
        last_unmatched_year=("year", "max"),
        unmatched_years=("year", "count"),
    )
    .merge(panel_coverage, on=["COUNTRY", "iso3"], how="left")
    .merge(defacto_coverage, on="iso3", how="left")
)

unmatched_audit["defacto_iso_present"] = unmatched_audit["defacto_first_year"].notna()
unmatched_audit["audit_reason"] = "matched key missing in specific years"
unmatched_audit.loc[~unmatched_audit["defacto_iso_present"], "audit_reason"] = "iso3 not present in defacto_all"
unmatched_audit.loc[
    unmatched_audit["iso3"].isin(aggregate_or_noncountry_codes),
    "audit_reason",
] = "aggregate/non-country code not in defacto data"
unmatched_audit.loc[
    unmatched_audit["defacto_iso_present"]
    & (unmatched_audit["last_unmatched_year"] < unmatched_audit["defacto_first_year"]),
    "audit_reason",
] = "panel years before defacto coverage starts"
unmatched_audit.loc[
    unmatched_audit["defacto_iso_present"]
    & (unmatched_audit["first_unmatched_year"] > unmatched_audit["defacto_last_year"]),
    "audit_reason",
] = "panel years after defacto coverage ends"
unmatched_audit.loc[
    unmatched_audit["defacto_iso_present"]
    & (unmatched_audit["first_unmatched_year"] < unmatched_audit["defacto_first_year"])
    & (unmatched_audit["last_unmatched_year"] >= unmatched_audit["defacto_first_year"]),
    "audit_reason",
] = "partial mismatch: early panel years before defacto coverage"
unmatched_audit.loc[
    unmatched_audit["defacto_iso_present"]
    & (unmatched_audit["first_unmatched_year"] <= unmatched_audit["defacto_last_year"])
    & (unmatched_audit["last_unmatched_year"] > unmatched_audit["defacto_last_year"]),
    "audit_reason",
] = "partial mismatch: late panel years after defacto coverage"

unmatched_audit["likely_aggregate_or_nonstandard_code"] = (
    unmatched_audit["iso3"].str.len().ne(3)
    | unmatched_audit["iso3"].isin(aggregate_or_noncountry_codes + ["XKX"])
)

unmatched_audit = unmatched_audit.sort_values(
    ["audit_reason", "unmatched_years", "COUNTRY"],
    ascending=[True, False, True],
)

unmatched_rows.to_csv(audit_country_year_path, index=False)
unmatched_audit.to_csv(audit_summary_path, index=False)

print(f"Saved unmatched country-years to {audit_country_year_path}")
print(f"Saved unmatched country summary to {audit_summary_path}")

Saved unmatched country-years to ../data/raw/defacto_unmatched_country_years.csv
Saved unmatched country summary to ../data/raw/defacto_unmatched_country_summary.csv


In [11]:
audit_reason_counts = (
    unmatched_audit.groupby("audit_reason", as_index=False)
    .agg(
        country_iso_pairs=("iso3", "count"),
        unmatched_country_years=("unmatched_years", "sum"),
    )
    .sort_values("unmatched_country_years", ascending=False)
)

display(audit_reason_counts)
display(unmatched_audit.head(30))

,audit_reason,country_iso_pairs,unmatched_country_years
2,panel years before defacto coverage starts,40,1029
0,aggregate/non-country code not in defacto data,4,528
1,matched key missing in specific years,3,59


,COUNTRY,iso3,first_unmatched_year,last_unmatched_year,unmatched_years,panel_first_year,panel_last_year,panel_years,defacto_country_name,defacto_first_year,defacto_last_year,defacto_years,defacto_iso_present,audit_reason,likely_aggregate_or_nonstandard_code
0,ASEAN,ASEAN,1894,2025,132,1894,2025,132,NaN,NaN,NaN,NaN,False,aggregate/non-country code not in defacto data,True
1,African Union,AU,1894,2025,132,1894,2025,132,NaN,NaN,NaN,NaN,False,aggregate/non-country code not in defacto data,True
13,European Union,EU,1894,2025,132,1894,2025,132,NaN,NaN,NaN,NaN,False,aggregate/non-country code not in defacto data,True
29,Organization of American States,OAS,1894,2025,132,1894,2025,132,NaN,NaN,NaN,NaN,False,aggregate/non-country code not in defacto data,True
32,Poland,POL,1894,1943,29,1894,2025,132,Poland,1789.0,2025.0,171.0,True,matched key missing in specific years,False
17,Honduras,HND,1921,1944,24,1894,2025,132,Honduras,1838.0,2025.0,188.0,True,matched key missing in specific years,False
28,Oman,OMN,1894,1899,6,1894,2025,132,Oman,1789.0,2025.0,237.0,True,matched key missing in specific years,False
37,South Sudan,SSD,1894,2010,117,1894,2025,132,South Sudan,2011.0,2025.0,15.0,True,panel years before defacto coverage starts,False
20,Kosovo,XKX,1894,1998,105,1894,2025,132,Kosovo,1999.0,2025.0,27.0,True,panel years before defacto coverage starts,True
4,Azerbaijan,AZE,1894,1989,96,1894,2025,132,Azerbaijan,1990.0,2025.0,36.0,True,panel years before defacto coverage starts,False
